# Исследовательский анализ данных (EDA)

Анализ датасета Brain_Tumor_Detect. Предполагается, что данные уже подготовлены
скриптом `python scripts/prepare_data.py --src <экспорт> --dst data/processed`.


In [ ]:
import os, yaml, cv2, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from src.dataset.dataset import yolo_to_xyxy

PROC = Path('data/processed')
names = yaml.safe_load(open(PROC/'data.yaml'))['names']
print('Классы:', names)


## Размеры выборок и распределение классов


In [ ]:
rows=[]
for split in ['train','val','test']:
    img_dir = PROC/split/'images'; lab_dir = PROC/split/'labels'
    if not img_dir.exists(): continue
    n_img = len([p for p in img_dir.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')])
    cnt = Counter()
    for lab in lab_dir.glob('*.txt'):
        for line in lab.read_text().strip().splitlines():
            if line.strip(): cnt[names[int(line.split()[0])]] += 1
    rows.append({'split':split,'images':n_img,**{n:cnt.get(n,0) for n in names}})
df = pd.DataFrame(rows); print(df.to_string(index=False))


In [ ]:
ax = df.set_index('split')[names].plot(kind='bar', figsize=(8,4))
ax.set_title('Распределение классов по выборкам'); ax.set_ylabel('объектов')
plt.tight_layout(); plt.show()


## Примеры снимков с разметкой


In [ ]:
img_dir = PROC/'train'/'images'; lab_dir = PROC/'train'/'labels'
imgs = random.sample([p for p in img_dir.iterdir()], 6)
fig, axes = plt.subplots(2,3, figsize=(14,9))
for ax, p in zip(axes.ravel(), imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h,w = im.shape[:2]
    lab = lab_dir/(p.stem+'.txt')
    if lab.exists():
        for line in lab.read_text().strip().splitlines():
            c,cx,cy,bw,bh = map(float, line.split()[:5])
            x1,y1,x2,y2 = yolo_to_xyxy(cx,cy,bw,bh,w,h)
            cv2.rectangle(im,(int(x1),int(y1)),(int(x2),int(y2)),(255,0,0),2)
            cv2.putText(im,names[int(c)],(int(x1),int(y1)-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,0,0),2)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()


## Распределение числа объектов на изображение


In [ ]:
counts=[]
for lab in (PROC/'train'/'labels').glob('*.txt'):
    counts.append(len([l for l in lab.read_text().strip().splitlines() if l.strip()]))
plt.figure(figsize=(7,4))
plt.hist(counts, bins=range(0, max(counts)+2), align='left', rwidth=0.8)
plt.xlabel('Объектов на изображении'); plt.ylabel('Число изображений')
plt.title('Распределение числа объектов на изображение (train)')
plt.tight_layout(); plt.show()
print('Среднее число объектов на изображение:', round(np.mean(counts),2))
